# Neural Network Based Text Mining for Mental Health Analysis
## BERT Model Training Notebook
**Author:** Muhammad Danial (297801)  
**Subject:** STIZK3993 - Data Science Project  
**Model:** bert-base-uncased → Fine-tuned for Mental Health Classification


In [1]:
# Install dependencies
!pip install transformers torch scikit-learn datasets pandas numpy matplotlib seaborn

# Install dependencies
!pip install -q transformers==4.40.0 datasets torch scikit-learn nltk matplotlib seaborn tensorflow keras imbalanced-learn wordcloud pandas numpy accelerate
print('✅ All packages installed!')

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------------- ---------------------- 3.7/8.3 MB 16.7 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.3 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 12.0 MB/s  0:00:00
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 8.4 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.9 MB 8.7 MB/s eta


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ All packages installed!


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import warnings
warnings.filterwarnings('ignore')

# ML & Deep Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, LSTM, Dropout

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


## 1. Dataset Preparation
Using the Reddit Mental Health dataset (public) with 6 labels.

In [ ]:
# 1. Load your specific dataset
df = pd.read_csv('Combined Data.csv')

# 2. Clean the data (Drop the unnamed index column and empty rows)
# The CSV has an unnamed index, 'statement', and 'status'
df = df.dropna(subset=['statement', 'status'])
df = df.rename(columns={'statement': 'text', 'status': 'label_text'})

# 3. Create a label mapping automatically based on YOUR dataset
unique_labels = df['label_text'].unique().tolist()
LABEL2ID = {label: i for i, label in enumerate(unique_labels)}
ID2LABEL = {i: label for i, label in enumerate(unique_labels)}

# 4. Apply mapping to the dataset
df['label'] = df['label_text'].map(LABEL2ID)

print("✅ Data Loaded! Found categories:", unique_labels)
print(f"Total rows: {len(df)}")

# Now you can proceed with train_test_split using df['text'] and df['label']


#-------------------------------------------
# Define paths to your datasets
# Update these paths to match your Google Drive directories
ENGLISH_DATA_PATH = '/content/drive/MyDrive/mental_health_english.csv' 
MALAY_DATA_PATH = '/content/drive/MyDrive/mental_health_malay.csv'

def clean_text(text):
    text = re.sub(r'\W', ' ', str(text).lower())
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

try:
    df_eng = pd.read_csv(ENGLISH_DATA_PATH)
    df_eng['language'] = 'en'
    
    df_mal = pd.read_csv(MALAY_DATA_PATH)
    df_mal['language'] = 'my'
    
    # Combine datasets
    df = pd.concat([df_eng, df_mal], ignore_index=True)

    EXPECTED = ["Normal","Depression","Suicidal","Anxiety","Bipolar","Stress","Personality disorder"]

    # Verify your dataset has matching labels
    print("Your labels:", df['label_text'].unique())
    print("Expected:   ", EXPECTED_LABELS)
    
    # Ensure columns are named correctly (assuming 'text' and 'label')
    df['text_clean'] = df['text'].apply(clean_text)
    
    print(f"Total Combined Samples: {len(df)}")
    print(df['language'].value_counts())
    
except Exception as e:
    print("Error loading data. Please ensure your CSV files are uploaded and paths are correct.")
    print(f"Error details: {e}")
    
    # Creating dummy data for testing the pipeline if files are missing
    print("\n--- Generating Dummy Data for Pipeline Testing ---")
    df = pd.DataFrame({
        'text_clean': ['I feel so sad and hopeless', 'Saya rasa sangat sedih dan putus asa', 'Life is going well', 'Hidup saya sangat gembira'],
        'label': [1, 1, 0, 0],
        'language': ['en', 'my', 'en', 'my']
    })

X_train, X_test, y_train, y_test = train_test_split(df['text_clean'], df['label'], test_size=0.2, random_state=42)

In [ ]:
# EDA - Label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_names = list(LABEL_MAP.keys())
counts = [df[df['label'] == i].shape[0] for i in range(NUM_LABELS)]

colors = ['#ef4444','#f97316','#eab308','#22c55e','#3b82f6','#8b5cf6']
axes[0].bar(label_names, counts, color=colors)
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('Mental Health Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(counts, labels=label_names, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Category Proportions', fontsize=14)

plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150)
plt.show()

## 2. BERT Tokenization & Dataset Class

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 7
LEARNING_RATE = 2e-5

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=SEED, stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=SEED
)

train_ds = MentalHealthDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = MentalHealthDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = MentalHealthDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 3. Model Architecture (BERT Fine-tuning)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL_MAP
)
model = model.to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Model: BERT-base-uncased + Linear Classification Head')
print(f'Hidden size: 768 | Layers: 12 | Heads: 12')

## 4. Training Loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

# Training
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('../models/best_bert_mental_health')
        tokenizer.save_pretrained('../models/best_bert_mental_health')
        print(f'  ✓ Best model saved (val_acc={val_acc:.4f})')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='#ef4444')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#3b82f6')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', color='#22c55e')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='#f97316')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/training_curves.png', dpi=150)
plt.show()

## 5. Evaluation & Metrics

In [ ]:
# Test set evaluation
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, device)
f1 = f1_score(test_labels, test_preds, average='weighted')

print('=' * 50)
print('TEST SET RESULTS')
print('=' * 50)
print(f'Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'F1 Score : {f1:.4f} (weighted)')
print(f'Loss     : {test_loss:.4f}')
print('\nDetailed Classification Report:')
print(classification_report(test_labels, test_preds, target_names=list(LABEL_MAP.keys())))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=list(LABEL_MAP.keys()),
    yticklabels=list(LABEL_MAP.keys())
)
plt.title('Confusion Matrix - BERT Mental Health Classifier', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=30)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150)
plt.show()

## 6. Inference Pipeline

In [ ]:
def predict_mental_health(text: str, model, tokenizer, device):
    """Run BERT inference on new text."""
    model.eval()
    encoding = tokenizer(
        text,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device)
        )
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    predicted_class = np.argmax(probs)
    label = ID2LABEL[predicted_class]

    print(f'Input: "{text[:80]}..."')
    print(f'Prediction: {label.upper()} (confidence: {probs[predicted_class]*100:.1f}%)')
    print('All probabilities:')
    for cat, prob in zip(LABEL_MAP.keys(), probs):
        bar = '█' * int(prob * 30)
        print(f'  {cat:15s} {bar} {prob*100:.1f}%')
    return label, probs

# Test inference
test_cases = [
    "I feel hopeless and cannot see any reason to continue.",
    "I am so anxious about everything, my heart races all the time.",
    "Life is good! I feel grateful and happy with my progress."
]
for tc in test_cases:
    print('\n' + '='*60)
    predict_mental_health(tc, model, tokenizer, device)